# Ingredient match debug — top-20 candidates with score breakdown

Uses the same staged matcher as [`food_mvp_recipe_matching.ipynb`](food_mvp_recipe_matching.ipynb) (`food_4macro` + cached embeddings).

Each query shows **ingredient-parser-nlp** fields (`quantity`, `unit`, `name`, `size`, `preparation`, …) then top-20 USDA matches with full **description** text (no truncation).

Requires cache under `scratch/recipe_matching_10k/` (run `ingredient_query_cache.py` or the main notebook first).

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "scripts"))

from ingredient_match_staged import (
    StagedMatchConfig,
    StagedFoodIndex,
    match_query_top_k,
    query_from_parsed_row,
)
from ingredient_query_cache import (
    embed_adhoc_recipe_queries,
    ensure_hf_token,
    load_or_build_food_artifacts,
)
from load_food_4macro import load_food_4macro
from parse_recipe_ingredient import PARSE_FIELDS
from progress_utils import force_std_tqdm
from recipe_match_cache import DEFAULT_CACHE_DIR

force_std_tqdm()
ensure_hf_token()

# Show full USDA descriptions (no "..." truncation in notebook output)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", None)
pd.set_option("display.max_columns", None)

WORK_DIR = DEFAULT_CACHE_DIR
MATCH_CONFIG = StagedMatchConfig()
TOP_K = 20

# ingredient-parser-nlp fields + pipeline extras
PARSED_DISPLAY_COLS = [
    "ingredient",
    *PARSE_FIELDS,
    "dequantified",
    "prep_used_unprepared",
]

/Users/danielcosta/.pyenv/versions/3.11.11/lib/python3.11/importlib/__init__.py:126: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  return _bootstrap._gcd_import(name[level:], package, level)


In [2]:
food_4macro_raw = load_food_4macro()
_, food_name_emb, food_prep_emb, food_dequant_emb, food_meta = load_or_build_food_artifacts(
    food_4macro_raw,
    WORK_DIR,
)
food_index = StagedFoodIndex.from_catalog(
    food_4macro_raw,
    name_embeddings=food_name_emb,
    prep_embeddings=food_prep_emb,
    dequant_embeddings=food_dequant_emb,
    config=MATCH_CONFIG,
    show_progress=False,
)
print(f"food index: {len(food_index.candidates):,} candidates")

Loaded cached food_4macro parse + embeddings (96,996 rows) → /Users/danielcosta/Berkeley/Capstone/scratch/recipe_matching_10k
food index: 96,996 candidates


In [3]:
examples = [
    "4 chicken breasts, skin removed",
    "1 Tbsp vanilla",
    "2 (16 oz.) cans lima beans",
]

parsed, name_emb, prep_emb, dequant_emb = embed_adhoc_recipe_queries(examples, WORK_DIR)

# ingredient-parser-nlp output (see scripts/parse_recipe_ingredient.py)
cols = [c for c in PARSED_DISPLAY_COLS if c in parsed.columns]
display(parsed[cols].T)  # transposed: one column per example, easier to read

Using HF_TOKEN from .env for Hugging Face Hub downloads.


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6548.83it/s]


Saved 'unprepared' embedding → /Users/danielcosta/Berkeley/Capstone/scratch/recipe_matching_10k/unprepared_prep_embedding.npy
Recipe prep: 2 lines use 'unprepared' proxy; 1 embed parsed preparation


,0,1,2
ingredient,"4 chicken breasts, skin removed",1 Tbsp vanilla,2 (16 oz.) cans lima beans
quantity,4.0,1.0,2.0
quantity_max,4.0,1.0,2.0
unit,NaN,Tbsp,cans
unit_raw,4,1 Tbsp,2 cans
amount_text,4,1 Tbsp,"2 cans, 16 oz"
name,chicken breasts,vanilla,lima beans
size,,,
preparation,skin removed,,
parse_status,ok,ok,ok


In [4]:
SCORE_COLS = [
    "rank",
    "fdc_id",
    "description",
    "match_score",
    "match_margin",
    "base_score",
    "prep_score",
    "default_bonus",
    "modifier_penalty",
    "name_lex_score",
    "name_sem_score",
    "dequant_lex_score",
    "dequant_sem_score",
    "name_channel_score",
    "dequant_channel_score",
    "prep_sem_score",
    "prep_lex_score",
    "prep_rules_score",
    "prep_default_component",
    "fallback_reason",
]


def display_full(df: pd.DataFrame) -> None:
    """Render DataFrame with untruncated text columns."""
    display(
        df.style.set_properties(
            subset=[c for c in df.columns if df[c].dtype == object],
            **{"white-space": "pre-wrap", "text-align": "left"},
        )
    )


def show_parser_fields(i: int) -> None:
    row = parsed.iloc[i]
    fields = [c for c in PARSED_DISPLAY_COLS if c in parsed.columns and c != "ingredient"]
    summary = pd.Series({c: row[c] for c in fields}, name=examples[i])
    display_full(summary.to_frame())


def show_top_k(i: int) -> pd.DataFrame:
    row = parsed.iloc[i]
    q = query_from_parsed_row(row, name_emb[i], prep_emb[i], dequant_emb[i])
    top = match_query_top_k(q, food_index, top_k=TOP_K, score_all_stage1=True)
    cols = [c for c in SCORE_COLS if c in top.columns]
    return top[cols]


for i, text in enumerate(examples):
    print("=" * 80)
    print(f"Query: {text}")
    print("\nParser fields (ingredient-parser-nlp):")
    show_parser_fields(i)
    print(f"\nTop {TOP_K} candidates:")
    display_full(show_top_k(i))

Query: 4 chicken breasts, skin removed

Parser fields (ingredient-parser-nlp):


,"4 chicken breasts, skin removed"
quantity,4.000000
quantity_max,4.000000
unit,nan
unit_raw,4
amount_text,4
name,chicken breasts
size,
preparation,skin removed
parse_status,ok
parse_method,ingredient_parser



Top 20 candidates:


,rank,fdc_id,description,match_score,match_margin,base_score,prep_score,default_bonus,modifier_penalty,name_lex_score,name_sem_score,dequant_lex_score,dequant_sem_score,name_channel_score,dequant_channel_score,prep_sem_score,prep_lex_score,prep_rules_score,prep_default_component,fallback_reason
0,1,360997,FRESH CHICKEN BREASTS,0.599800,0.008400,0.629600,0.362300,1.000000,0.000000,0.667100,0.861500,0.399500,0.658000,0.706000,0.451200,0.155700,0.000000,1.000000,1.000000,base_only
1,2,360613,BONELESS BREAST OF CHICKEN,0.591400,0.013800,0.616700,0.362300,1.000000,0.000000,0.665000,0.739400,0.408300,0.712500,0.679900,0.469200,0.155700,0.000000,1.000000,1.000000,base_only
2,3,1123173,"BONELESS, SKINLESS CHICKEN BREASTS",0.577600,0.003700,0.595400,0.362300,1.000000,0.000000,0.637500,0.755000,0.379000,0.696000,0.661000,0.442400,0.155700,0.000000,1.000000,1.000000,base_only
3,4,394201,RAW STUFFED CHICKEN BREASTS,0.573900,0.018100,0.589800,0.362300,1.000000,0.000000,0.639600,0.736100,0.382700,0.611200,0.658900,0.428400,0.155700,0.000000,1.000000,1.000000,base_only
4,5,1133051,"CHICKEN BREAST BONELESS SKINLESS WITH RIB MEAT, CHICKEN BREAST",0.555800,0.002300,0.561900,0.362300,1.000000,0.000000,0.609800,0.665900,0.349000,0.724000,0.621100,0.424000,0.155700,0.000000,1.000000,1.000000,base_only
5,6,1121566,"BONELESS, SKINLESS CHICKEN BREAST WITH RIB MEAT",0.553500,0.000200,0.558300,0.362300,1.000000,0.000000,0.609800,0.671700,0.349000,0.650500,0.622200,0.409300,0.155700,0.000000,1.000000,1.000000,base_only
6,7,1128266,"BONELESS, SKINLESS CHICKEN BREAST TENDERLOINS",0.553300,0.000700,0.558100,0.362300,1.000000,0.000000,0.620200,0.617800,0.359700,0.633600,0.619700,0.414400,0.155700,0.000000,1.000000,1.000000,base_only
7,8,1120622,"DICED CHICKEN BREASTS BONELESS, SKINLESS WITH RIB MEAT, DICED CHICKEN BREASTS",0.552600,0.008200,0.556900,0.362300,1.000000,0.000000,0.601600,0.721100,0.339500,0.626300,0.625500,0.396900,0.155700,0.000000,1.000000,1.000000,base_only
8,9,358498,"PERDUE, FIT & EASY, BONELESS SKINLESS CHICKEN BREASTS",0.544400,0.004900,0.544300,0.362300,1.000000,0.000000,0.601800,0.629400,0.340100,0.626300,0.607300,0.397300,0.155700,0.000000,1.000000,1.000000,base_only
9,10,1120618,"GRILLED CHICKEN BREAST STRIPS BONELESS SKINLESS WITH RIB MEAT, GRILLED CHICKEN BREAST",0.539500,0.007400,0.536900,0.362300,1.000000,0.000000,0.595400,0.602500,0.332500,0.655000,0.596800,0.397000,0.155700,0.000000,1.000000,1.000000,base_only


Query: 1 Tbsp vanilla

Parser fields (ingredient-parser-nlp):


...

,rank,fdc_id,description,match_score,match_margin,base_score,prep_score,default_bonus,modifier_penalty,name_lex_score,name_sem_score,dequant_lex_score,dequant_sem_score,name_channel_score,dequant_channel_score,prep_sem_score,prep_lex_score,prep_rules_score,prep_default_component,fallback_reason
0,1,1129775,"VANILLA SHAKE, VANILLA",0.584200,0.001300,0.592500,0.436200,0.900000,0.000000,0.625000,0.815100,0.373100,0.647200,0.663000,0.427900,0.240400,0.500000,0.500000,0.900000,neutral_default
1,2,1132007,"VANILLA POWDER, VANILLA",0.582900,0.001600,0.590500,0.436200,0.900000,0.000000,0.625000,0.797700,0.372200,0.658900,0.659500,0.429500,0.240400,0.500000,0.500000,0.900000,neutral_default
2,3,1120016,"VANILLA SYRUP, VANILLA",0.581300,0.003800,0.588100,0.436200,0.900000,0.000000,0.625000,0.795800,0.373100,0.619100,0.659200,0.422300,0.240400,0.500000,0.500000,0.900000,neutral_default
3,4,384417,VANILLA BEAN,0.577500,0.000400,0.582200,0.436200,0.900000,0.000000,0.625000,0.752000,0.375000,0.615500,0.650400,0.423100,0.240400,0.500000,0.500000,0.900000,neutral_default
4,5,412952,VANILLA COFFEE,0.577100,0.000100,0.581600,0.436200,0.900000,0.000000,0.625000,0.757800,0.373100,0.600100,0.651600,0.418500,0.240400,0.500000,0.500000,0.900000,neutral_default
5,6,1113753,"COLA, VANILLA",0.577000,0.001300,0.581400,0.436200,0.900000,0.000000,0.625000,0.761900,0.374000,0.583400,0.652400,0.415900,0.240400,0.500000,0.500000,0.900000,neutral_default
6,7,1121528,"VANILLA GELATO, VANILLA",0.575700,0.000100,0.579400,0.436200,0.900000,0.000000,0.625000,0.743400,0.372200,0.599800,0.648700,0.417700,0.240400,0.500000,0.500000,0.900000,neutral_default
7,8,1112960,"VANILLA PUDDING, VANILLA",0.575600,0.001800,0.579300,0.436200,0.900000,0.000000,0.625000,0.722300,0.371400,0.650600,0.644500,0.427300,0.240400,0.500000,0.500000,0.900000,neutral_default
8,9,398095,VANILLA BEANS,0.573800,0.004400,0.576500,0.436200,0.900000,0.000000,0.625000,0.720400,0.374000,0.598900,0.644100,0.419000,0.240400,0.500000,0.500000,0.900000,neutral_default
9,10,1135749,"VANILLA ICE CREAM, VANILLA",0.569400,0.000300,0.569800,0.436200,0.900000,0.000000,0.600000,0.807700,0.345000,0.631100,0.641500,0.402200,0.240400,0.500000,0.500000,0.900000,neutral_default


Query: 2 (16 oz.) cans lima beans

Parser fields (ingredient-parser-nlp):


,2 (16 oz.) cans lima beans
quantity,2.000000
quantity_max,2.000000
unit,cans
unit_raw,2 cans
amount_text,"2 cans, 16 oz"
name,lima beans
size,
preparation,
parse_status,ok
parse_method,ingredient_parser



Top 20 candidates:


,rank,fdc_id,description,match_score,match_margin,base_score,prep_score,default_bonus,modifier_penalty,name_lex_score,name_sem_score,dequant_lex_score,dequant_sem_score,name_channel_score,dequant_channel_score,prep_sem_score,prep_lex_score,prep_rules_score,prep_default_component,fallback_reason
0,1,1117820,LIMA BEANS,0.652300,0.004500,0.697300,0.436200,0.900000,0.000000,0.731600,1.000000,0.444400,0.683300,0.785300,0.492100,0.240400,0.500000,0.500000,0.900000,neutral_default
1,2,168396,"Lima beans, immature seeds, raw",0.647800,0.008300,0.658400,0.479200,1.000000,0.000000,0.730000,0.722500,0.461900,0.626700,0.728500,0.494900,0.322900,0.500000,0.500000,1.000000,neutral_default
2,3,174252,"Lima beans, large, mature seeds, raw",0.639500,0.023200,0.645600,0.479200,1.000000,0.000000,0.706700,0.733500,0.451300,0.648600,0.712000,0.490800,0.322900,0.500000,0.500000,1.000000,neutral_default
3,4,390919,GREEN LIMA BEANS,0.616300,0.000500,0.641900,0.436200,0.900000,0.000000,0.674000,0.909700,0.407400,0.656100,0.721100,0.457200,0.240400,0.500000,0.500000,0.900000,neutral_default
4,5,383741,SMALL LIMA BEANS,0.615800,0.001600,0.641200,0.436200,0.900000,0.000000,0.674000,0.879200,0.407400,0.715400,0.715000,0.469000,0.240400,0.500000,0.500000,0.900000,neutral_default
5,6,1135568,LARGE LIMA BEANS,0.614200,0.004800,0.638700,0.436200,0.900000,0.000000,0.674000,0.865500,0.407400,0.704700,0.712300,0.466900,0.240400,0.500000,0.500000,0.900000,neutral_default
6,7,390890,FRESH CUT LIMA BEANS,0.609400,0.013200,0.612000,0.446200,1.000000,0.000000,0.645700,0.823500,0.393900,0.677100,0.681300,0.450500,0.240400,0.500000,0.500000,1.000000,neutral_default
7,8,1131261,GIANT PERUVIAN LIMA BEANS,0.596200,0.000900,0.611000,0.436200,0.900000,0.000000,0.642600,0.832000,0.389100,0.687400,0.680500,0.448800,0.240400,0.500000,0.500000,0.900000,neutral_default
8,9,382124,PREMIUM BABY LIMA BEANS,0.595300,0.000300,0.609600,0.436200,0.900000,0.000000,0.643800,0.840000,0.390900,0.628400,0.683000,0.438400,0.240400,0.500000,0.500000,0.900000,neutral_default
9,10,1120032,DELUXE TINY LIMA BEANS,0.595000,0.002600,0.609200,0.436200,0.900000,0.000000,0.644400,0.817500,0.391900,0.665000,0.679000,0.446500,0.240400,0.500000,0.500000,0.900000,neutral_default
